# 05_01 - LightGBM Raw 777

Train and evaluate LightGBM on ModernBERT pooled embeddings plus 9 behavioral features.


## Contract

Run in Colab only. Read Phase 2 raw 777 artifacts. Write probabilities, metrics and metadata for downstream ensemble notebooks.

- **Multi-seed:** set `SEEDS = [42, 123, 456]` in the setup cell; one run trains all seeds into isolated artifact subfolders (`seed_123`, `seed_456`). Seed `42` keeps canonical paths.


In [1]:
# try/except: khối xử lý ngoại lệ
try:
    # import google.colab  # type: ignore: import thư viện google
    import google.colab  # type: ignore
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = True
# except: xử lý ngoại lệ — except ImportError:
except ImportError:
    # IN_COLAB: biến cấu hình/hằng số của notebook
    IN_COLAB = False

# if: điều kiện — if not IN_COLAB:
if not IN_COLAB:
    # raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 tr...: ném lỗi và dừng cell
    raise RuntimeError("Run this notebook in Google Colab. Do not execute Phase 5 training locally.")


In [2]:
# from google.colab import drive: import thư viện google
from google.colab import drive
# drive.mount("/content/drive"): mount Google Drive trên Colab
drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
# import gc: giải phóng bộ nhớ
import gc
# import importlib: import thư viện importlib
import importlib
# import json: đọc/ghi JSON metadata
import json
# import os: biến môi trường hệ thống
import os
# import random: cố định seed ngẫu nhiên
import random
# import subprocess: chạy lệnh pip/cài package
import subprocess
# import sys: tham số Python runtime
import sys
# import time: đo thời gian thực thi
import time
# from datetime import datetime, timezone: import thư viện datetime
from datetime import datetime, timezone
# from itertools import combinations, product: import thư viện itertools
from itertools import combinations, product
# from pathlib import Path: quản lý đường dẫn
from pathlib import Path

# import joblib: lưu/tải object đã fit
import joblib
# import numpy: tính toán mảng số
import numpy as np
# import pandas: xử lý DataFrame
import pandas as pd
# from sklearn.metrics import (: thư viện machine learning scikit-learn
from sklearn.metrics import (
    # accuracy_score,: thực thi lệnh Python
    accuracy_score,
    # average_precision_score,: thực thi lệnh Python
    average_precision_score,
    # brier_score_loss,: thực thi lệnh Python
    brier_score_loss,
    # confusion_matrix,: thực thi lệnh Python
    confusion_matrix,
    # f1_score,: thực thi lệnh Python
    f1_score,
    # precision_recall_fscore_support,: thực thi lệnh Python
    precision_recall_fscore_support,
    # roc_auc_score,: thực thi lệnh Python
    roc_auc_score,
# ): đóng ngoặc gọi hàm
)

# SEEDS: biến cấu hình/hằng số của notebook
SEEDS = [42, 123, 456]
# SEED: biến cấu hình/hằng số của notebook
SEED = SEEDS[0]
# FAKE_LABEL: biến cấu hình/hằng số của notebook
FAKE_LABEL = 1
# REAL_LABEL: biến cấu hình/hằng số của notebook
REAL_LABEL = 0
# DEFAULT_THRESHOLD: biến cấu hình/hằng số của notebook
DEFAULT_THRESHOLD = 0.50
# TARGET_PRECISION_FAKE: biến cấu hình/hằng số của notebook
TARGET_PRECISION_FAKE = 0.975

# PROJECT_ROOT: biến cấu hình/hằng số của notebook
PROJECT_ROOT = Path(os.environ.get("FAKE_REVIEWS_PROJECT_ROOT", "/content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews"))
# FEATURE_DIR: biến cấu hình/hằng số của notebook
FEATURE_DIR = PROJECT_ROOT / "artifacts" / "features"
# MODEL_DIR: biến cấu hình/hằng số của notebook
MODEL_DIR = PROJECT_ROOT / "artifacts" / "models"
# ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
ENSEMBLE_DIR = PROJECT_ROOT / "artifacts" / "ensemble"
# PREDICTION_DIR: biến cấu hình/hằng số của notebook
PREDICTION_DIR = PROJECT_ROOT / "artifacts" / "predictions"
# REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
# REPORT_FIGURE_DIR: biến cấu hình/hằng số của notebook
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"
# PROCESSED_DIR: biến cấu hình/hằng số của notebook
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# for: vòng lặp — for directory in [REPORT_FIGURE_DIR]:
for directory in [REPORT_FIGURE_DIR]:
    # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
    directory.mkdir(parents=True, exist_ok=True)

# RAW_FEATURE_PATHS: biến cấu hình/hằng số của notebook
RAW_FEATURE_PATHS = {
    # "train": FEATURE_DIR / "features_raw_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "features_raw_train.npy",
    # "val": FEATURE_DIR / "features_raw_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "features_raw_val.npy",
    # "test": FEATURE_DIR / "features_raw_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "features_raw_test.npy",
# }: đóng khối từ điển
}
# LABEL_PATHS: biến cấu hình/hằng số của notebook
LABEL_PATHS = {
    # "train": FEATURE_DIR / "labels_train.npy",: thực thi lệnh Python
    "train": FEATURE_DIR / "labels_train.npy",
    # "val": FEATURE_DIR / "labels_val.npy",: thực thi lệnh Python
    "val": FEATURE_DIR / "labels_val.npy",
    # "test": FEATURE_DIR / "labels_test.npy",: thực thi lệnh Python
    "test": FEATURE_DIR / "labels_test.npy",
# }: đóng khối từ điển
}
# FEATURE_METADATA_PATH: biến cấu hình/hằng số của notebook
FEATURE_METADATA_PATH = FEATURE_DIR / "feature_metadata.json"


# configure_seed_artifacts: thiết lập đường dẫn artifact theo seed
def configure_seed_artifacts(seed: int) -> int:
    # """Canonical paths for seed 42; isolated subfolders for other seeds.""": thực thi lệnh Python
    """Canonical paths for seed 42; isolated subfolders for other seeds."""
    # global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR: thực thi lệnh Python
    global SEED, PREDICTION_DIR, ENSEMBLE_DIR, REPORT_TABLE_DIR, MODEL_DIR
    # SEED: biến cấu hình/hằng số của notebook
    SEED = int(seed)
    # if: điều kiện — if SEED == 42:
    if SEED == 42:
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions"
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble"
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables"
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models"
    # else: nhánh còn lại của điều kiện
    else:
        # tag = ...: gán giá trị cho biến tag
        tag = f"seed_{SEED}"
        # prediction_dir = ...: dự đoán nhãn/xác suất
        prediction_dir = PROJECT_ROOT / "artifacts" / "predictions" / tag
        # ensemble_dir = ...: gán giá trị cho biến ensemble dir
        ensemble_dir = PROJECT_ROOT / "artifacts" / "ensemble" / tag
        # report_table_dir = ...: gán giá trị cho biến report table dir
        report_table_dir = PROJECT_ROOT / "reports" / "tables" / tag
        # model_dir = ...: gán giá trị cho biến model dir
        model_dir = PROJECT_ROOT / "artifacts" / "models" / tag
    # for: vòng lặp — for directory in [prediction_dir, ensemble_dir, report_table
    for directory in [prediction_dir, ensemble_dir, report_table_dir, model_dir]:
        # directory.mkdir(parents=True, exist_ok=True): tạo thư mục nếu chưa có
        directory.mkdir(parents=True, exist_ok=True)
    # PREDICTION_DIR: biến cấu hình/hằng số của notebook
    PREDICTION_DIR = prediction_dir
    # ENSEMBLE_DIR: biến cấu hình/hằng số của notebook
    ENSEMBLE_DIR = ensemble_dir
    # REPORT_TABLE_DIR: biến cấu hình/hằng số của notebook
    REPORT_TABLE_DIR = report_table_dir
    # MODEL_DIR: biến cấu hình/hằng số của notebook
    MODEL_DIR = model_dir
    # return: trả kết quả từ hàm
    return SEED


# set_global_seed: cố định seed random và numpy
def set_global_seed(seed: int) -> None:
    # random.seed(seed): cố định seed random
    random.seed(seed)
    # np.random.seed(seed): cố định seed numpy
    np.random.seed(seed)


# set_torch_seed: hàm xử lý set torch seed
def set_torch_seed(seed: int) -> None:
    # import torch: deep learning PyTorch
    import torch

    # torch.manual_seed(seed): cố định seed torch
    torch.manual_seed(seed)
    # if: điều kiện — if torch.cuda.is_available():
    if torch.cuda.is_available():
        # torch.cuda.manual_seed_all(seed): thực thi lệnh Python
        torch.cuda.manual_seed_all(seed)


# configure_seed_artifacts(SEEDS[0]): thực thi lệnh Python
configure_seed_artifacts(SEEDS[0])
# set_global_seed(SEEDS[0]): tạo tập hợp
set_global_seed(SEEDS[0])


# utc_now: định nghĩa hàm utc now
def utc_now() -> str:
    # return: trả kết quả từ hàm
    return datetime.now(timezone.utc).isoformat()


# ensure_package: import hoặc pip install package
def ensure_package(import_name: str, pip_name: str | None = None):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)
    # except: xử lý ngoại lệ — except ImportError:
    except ImportError:
        # subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or impor...: thực thi lệnh Python
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name or import_name])
        # return: trả kết quả từ hàm
        return importlib.import_module(import_name)


# read_json: đọc file JSON
def read_json(path: Path, default=None):
    # if: điều kiện — if not path.exists():
    if not path.exists():
        # return: trả kết quả từ hàm
        return default if default is not None else {}
    # with: context manager — with path.open("r", encoding="utf-8") as file:
    with path.open("r", encoding="utf-8") as file:
        # return: parse nội dung JSON
        return json.load(file)


# load_raw_arrays: nạp feature/label .npy từ Phase 2
def load_raw_arrays():
    # missing = ...: kiểm tra file/thư mục tồn tại
    missing = [str(path) for path in list(RAW_FEATURE_PATHS.values()) + list(LABEL_PATHS.values()) if not path.exists()]
    # if: điều kiện — if missing:
    if missing:
        # raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(miss...: ghi dictionary ra JSON
        raise FileNotFoundError("Missing Phase 2 raw feature inputs: " + json.dumps(missing, indent=2))
    # X: biến cấu hình/hằng số của notebook
    X = {split: np.load(path).astype(np.float32, copy=False) for split, path in RAW_FEATURE_PATHS.items()}
    # y = ...: nạp mảng từ file .npy
    y = {split: np.load(path).astype(np.int64, copy=False) for split, path in LABEL_PATHS.items()}
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # if: điều kiện — if X[split].ndim != 2:
        if X[split].ndim != 2:
            # raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}"): ném lỗi và dừng cell
            raise ValueError(f"{split} raw features must be 2D, got {X[split].shape}")
        # if: điều kiện — if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
        if y[split].ndim != 1 or len(y[split]) != X[split].shape[0]:
            # raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[spl...: ném lỗi và dừng cell
            raise ValueError(f"{split} label/features mismatch: X={X[split].shape}, y={y[split].shape}")
        # if: điều kiện — if not np.isfinite(X[split]).all():
        if not np.isfinite(X[split]).all():
            # raise ValueError(f"Non-finite raw features in {split}"): ném lỗi và dừng cell
            raise ValueError(f"Non-finite raw features in {split}")
    # return: trả kết quả từ hàm
    return X, y


# safe_roc_auc: ROC-AUC an toàn
def safe_roc_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(roc_auc_score(y_true, prob_fake))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# safe_pr_auc: PR-AUC an toàn
def safe_pr_auc(y_true, prob_fake):
    # try/except: khối xử lý ngoại lệ
    try:
        # return: trả kết quả từ hàm
        return float(average_precision_score(y_true, prob_fake, pos_label=FAKE_LABEL))
    # except: xử lý ngoại lệ — except ValueError:
    except ValueError:
        # return: trả kết quả từ hàm
        return float("nan")


# evaluate_predictions: tính metric classification từ xác suất
def evaluate_predictions(y_true, prob_fake, split, model_variant, threshold=DEFAULT_THRESHOLD, threshold_strategy="default_0.5"):
    # y_true = ...: ép kiểu dữ liệu cột
    y_true = np.asarray(y_true).astype(int)
    # prob_fake = ...: ép kiểu dữ liệu cột
    prob_fake = np.asarray(prob_fake).astype(float)
    # y_pred = ...: ép kiểu dữ liệu cột
    y_pred = (prob_fake >= threshold).astype(int)
    # precision, recall, f1, support = precision_recall_fscore_support(: thực thi lệnh Python
    precision, recall, f1, support = precision_recall_fscore_support(
        # y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0: thực thi lệnh Python
        y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL], zero_division=0
    # ): đóng ngoặc gọi hàm
    )
    # tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL...: thực thi lệnh Python
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[REAL_LABEL, FAKE_LABEL]).ravel()
    # return: trả kết quả từ hàm
    return {
        # "generated_at_utc": utc_now(),: thực thi lệnh Python
        "generated_at_utc": utc_now(),
        # "seed": int(SEED),: ép kiểu số nguyên
        "seed": int(SEED),
        # "split": split,: thực thi lệnh Python
        "split": split,
        # "model_variant": model_variant,: thực thi lệnh Python
        "model_variant": model_variant,
        # "threshold": float(threshold),: ép kiểu số thực
        "threshold": float(threshold),
        # "threshold_strategy": threshold_strategy,: ép kiểu chuỗi
        "threshold_strategy": threshold_strategy,
        # "accuracy": float(accuracy_score(y_true, y_pred)),: ép kiểu số thực
        "accuracy": float(accuracy_score(y_true, y_pred)),
        # "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),: ép kiểu số thực
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        # "precision_fake": float(precision[1]),: ép kiểu số thực
        "precision_fake": float(precision[1]),
        # "recall_fake": float(recall[1]),: ép kiểu số thực
        "recall_fake": float(recall[1]),
        # "f1_fake": float(f1[1]),: ép kiểu số thực
        "f1_fake": float(f1[1]),
        # "support_real": int(support[0]),: ép kiểu số nguyên
        "support_real": int(support[0]),
        # "support_fake": int(support[1]),: ép kiểu số nguyên
        "support_fake": int(support[1]),
        # "roc_auc": safe_roc_auc(y_true, prob_fake),: thực thi lệnh Python
        "roc_auc": safe_roc_auc(y_true, prob_fake),
        # "pr_auc": safe_pr_auc(y_true, prob_fake),: thực thi lệnh Python
        "pr_auc": safe_pr_auc(y_true, prob_fake),
        # "brier_score": float(brier_score_loss(y_true, prob_fake)),: ép kiểu số thực
        "brier_score": float(brier_score_loss(y_true, prob_fake)),
        # "tn": int(tn),: ép kiểu số nguyên
        "tn": int(tn),
        # "fp": int(fp),: ép kiểu số nguyên
        "fp": int(fp),
        # "fn": int(fn),: ép kiểu số nguyên
        "fn": int(fn),
        # "tp": int(tp),: ép kiểu số nguyên
        "tp": int(tp),
    # }: đóng khối từ điển
    }


# save_probability: lưu xác suất fake ra file .npy
def save_probability(prob_fake, model_variant, split):
    # path = ...: gán giá trị cho biến path
    path = PREDICTION_DIR / f"{model_variant}_{split}_prob.npy"
    # np.save(path, np.asarray(prob_fake, dtype=np.float32)): lưu mảng numpy ra file .npy
    np.save(path, np.asarray(prob_fake, dtype=np.float32))
    # return: trả kết quả từ hàm
    return str(path)


# write_metrics: ghi bảng metric ra CSV
def write_metrics(prob_map, y, model_variant, output_name):
    # rows = ...: gán giá trị cho biến rows
    rows = []
    # for: vòng lặp — for split in ["train", "val", "test"]:
    for split in ["train", "val", "test"]:
        # row = ...: dự đoán nhãn/xác suất
        row = evaluate_predictions(y[split], prob_map[split], split=split, model_variant=model_variant)
        # row["probability_path"] = save_probability(prob_map[split], model_variant, split...: thực thi lệnh Python
        row["probability_path"] = save_probability(prob_map[split], model_variant, split)
        # rows.append(row): thực thi lệnh Python
        rows.append(row)
    # df = ...: gán giá trị cho biến df
    df = pd.DataFrame(rows)
    # path = ...: gán giá trị cho biến path
    path = REPORT_TABLE_DIR / output_name
    # df.to_csv(path, index=False): ghi DataFrame ra file CSV
    df.to_csv(path, index=False)
    # display(df): hiển thị bảng/kết quả trên notebook
    display(df)
    # print("Saved metrics:", path): in thông tin ra console
    print("Saved metrics:", path)
    # return: trả kết quả từ hàm
    return df, path


In [4]:
# lightgbm = ...: gán giá trị cho biến lightgbm
lightgbm = ensure_package("lightgbm")
# from lightgbm import LGBMClassifier: thư viện LightGBM
from lightgbm import LGBMClassifier

# X, y = load_raw_arrays(): thực thi lệnh Python
X, y = load_raw_arrays()
# print({split: X[split].shape for split in X}): in thông tin ra console
print({split: X[split].shape for split in X})


{'train': (29923, 777), 'val': (6413, 777), 'test': (6413, 777)}


In [5]:
# for: vòng lặp — for SEED in SEEDS:
for SEED in SEEDS:
    # configure_seed_artifacts(SEED): thực thi lệnh Python
    configure_seed_artifacts(SEED)
    # set_global_seed(SEED): tạo tập hợp
    set_global_seed(SEED)
    # print(f"\n=== seed={SEED} ==="): in thông tin ra console
    print(f"\n=== seed={SEED} ===")
    # MODEL_VARIANT: biến cấu hình/hằng số của notebook
    MODEL_VARIANT = "phase5_lgbm_raw"
    # CONFIG: biến cấu hình/hằng số của notebook
    CONFIG = {
        # "n_estimators": 700,: thực thi lệnh Python
        "n_estimators": 700,
        # "num_leaves": 63,: thực thi lệnh Python
        "num_leaves": 63,
        # "learning_rate": 0.035,: thực thi lệnh Python
        "learning_rate": 0.035,
        # "subsample": 0.90,: thực thi lệnh Python
        "subsample": 0.90,
        # "colsample_bytree": 0.75,: thực thi lệnh Python
        "colsample_bytree": 0.75,
        # "min_child_samples": 20,: lấy giá trị nhỏ nhất
        "min_child_samples": 20,
        # "reg_alpha": 0.10,: thực thi lệnh Python
        "reg_alpha": 0.10,
        # "reg_lambda": 1.00,: thực thi lệnh Python
        "reg_lambda": 1.00,
        # "objective": "binary",: thực thi lệnh Python
        "objective": "binary",
        # "random_state": SEED,: thực thi lệnh Python
        "random_state": SEED,
        # "n_jobs": -1,: thực thi lệnh Python
        "n_jobs": -1,
        # "verbosity": -1,: thực thi lệnh Python
        "verbosity": -1,
    # }: đóng khối từ điển
    }
    # start_time = ...: gán giá trị cho biến start time
    start_time = time.time()
    # model = ...: xóa biến để giải phóng RAM/VRAM
    model = LGBMClassifier(**CONFIG)
    # model.fit(X["train"], y["train"], eval_set=[(X["val"], y["val"])]): fit model/reducer trên dữ liệu train
    model.fit(X["train"], y["train"], eval_set=[(X["val"], y["val"])])
    # train_seconds = ...: làm tròn số
    train_seconds = round(time.time() - start_time, 3)
    # model_path = ...: gán giá trị cho biến model path
    model_path = ENSEMBLE_DIR / f"{MODEL_VARIANT}.joblib"
    # joblib.dump({"model": model, "config": CONFIG, "seed": SEED, "train_seconds": tr...: lưu object (scaler/model) ra disk
    joblib.dump({"model": model, "config": CONFIG, "seed": SEED, "train_seconds": train_seconds}, model_path)
    # prob_map = ...: dự đoán nhãn/xác suất
    prob_map = {split: model.predict_proba(X[split])[:, FAKE_LABEL].astype(np.float32) for split in ["train", "val", "test"]}
    # metrics_df, metrics_path = write_metrics(prob_map, y, MODEL_VARIANT, "phase5_lgb...: thực thi lệnh Python
    metrics_df, metrics_path = write_metrics(prob_map, y, MODEL_VARIANT, "phase5_lgbm_raw_metrics.csv")
    # with: context manager — with (ENSEMBLE_DIR / f"{MODEL_VARIANT}_metadata.json").open(
    with (ENSEMBLE_DIR / f"{MODEL_VARIANT}_metadata.json").open("w", encoding="utf-8") as file:
        # json.dump({"model_variant": MODEL_VARIANT, "input_source": "phase2_raw_777", "mo...: ghi dictionary ra JSON
        json.dump({"model_variant": MODEL_VARIANT, "input_source": "phase2_raw_777", "model_path": str(model_path), "metrics_path": str(metrics_path), "train_seconds": train_seconds, "config": CONFIG}, file, indent=2)
    # print("Saved model:", model_path): in thông tin ra console
    print("Saved model:", model_path)
    # gc.collect(): giải phóng bộ nhớ
    gc.collect()



=== seed=42 ===


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T08:37:29.241900+00:00,42,train,phase5_lgbm_raw,0.5,default_0.5,1.000000,1.000000,1.000000,1.000000,...,17677,12246,1.000000,1.000000,0.001101,17677,0,0,12246,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T08:37:29.874889+00:00,42,val,phase5_lgbm_raw,0.5,default_0.5,0.907843,0.901611,0.967356,0.801829,...,3789,2624,0.955587,0.954939,0.075030,3718,71,520,2104,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T08:37:30.261787+00:00,42,test,phase5_lgbm_raw,0.5,default_0.5,0.910962,0.905099,0.967654,0.809451,...,3789,2624,0.954835,0.955677,0.073254,3718,71,500,2124,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/phase5_lgbm_raw_metrics.csv
Saved model: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/ensemble/phase5_lgbm_raw.joblib

=== seed=123 ===


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T08:42:50.607466+00:00,123,train,phase5_lgbm_raw,0.5,default_0.5,1.000000,1.000000,1.000000,1.000000,...,17677,12246,1.000000,1.000000,0.001118,17677,0,0,12246,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T08:42:50.638409+00:00,123,val,phase5_lgbm_raw,0.5,default_0.5,0.908935,0.902834,0.967461,0.804497,...,3789,2624,0.955426,0.954969,0.075032,3718,71,513,2111,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T08:42:50.660949+00:00,123,test,phase5_lgbm_raw,0.5,default_0.5,0.912677,0.906935,0.969945,0.811738,...,3789,2624,0.954242,0.955485,0.073370,3723,66,494,2130,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/seed_123/phase5_lgbm_raw_metrics.csv
Saved model: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/ensemble/seed_123/phase5_lgbm_raw.joblib

=== seed=456 ===


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,generated_at_utc,seed,split,model_variant,threshold,threshold_strategy,accuracy,macro_f1,precision_fake,recall_fake,...,support_real,support_fake,roc_auc,pr_auc,brier_score,tn,fp,fn,tp,probability_path
0,2026-06-10T08:48:10.963169+00:00,456,train,phase5_lgbm_raw,0.5,default_0.5,1.000000,1.000000,1.000000,1.000000,...,17677,12246,1.000000,1.00000,0.001105,17677,0,0,12246,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
1,2026-06-10T08:48:11.202953+00:00,456,val,phase5_lgbm_raw,0.5,default_0.5,0.908623,0.902419,0.969153,0.802210,...,3789,2624,0.954584,0.95443,0.074865,3722,67,519,2105,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...
2,2026-06-10T08:48:11.225426+00:00,456,test,phase5_lgbm_raw,0.5,default_0.5,0.910806,0.904893,0.968493,0.808308,...,3789,2624,0.954359,0.95548,0.073360,3720,69,503,2121,/content/drive/MyDrive/BaoMatCuoiKy/Fake_revie...


Saved metrics: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/reports/tables/seed_456/phase5_lgbm_raw_metrics.csv
Saved model: /content/drive/MyDrive/BaoMatCuoiKy/Fake_reviews/artifacts/ensemble/seed_456/phase5_lgbm_raw.joblib
